In [1]:
import os

os.environ["OPENAI_API_KEY"] = "REPLACE_ME" # Replace with actual OpenAI API key
os.environ["HF_TOKEN"] = "REPLACE_ME" # Replace with actual hugging face API key

In [2]:
# Reference: https://docs.langchain.com/oss/python/langchain/guardrails

!pip install langchain langchain-openai langgraph -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.0/503.0 kB 12.8 MB/s eta 0:00:00


#PII Detection

In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_openai import ChatOpenAI

# A simple no-op tool for demo purposes
from langchain.tools import tool

@tool
def echo_tool(text: str) -> str:
    """Echoes back the input text."""
    return f"Echo: {text}"

agent = create_agent(
    model="gpt-4.1-mini",
    tools=[echo_tool],
    middleware=[
        # Emails in user input → replaced with [REDACTED_EMAIL]
        PIIMiddleware("email", strategy="redact", apply_to_input=True),

        # Credit cards → masked (e.g. ****-****-****-1234)
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),

        # Custom API key pattern → raises an error immediately
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32,}",
            strategy="block",
            apply_to_input=True,
        ),
    ],
)

# Test 1: Email redaction
result = agent.invoke({
    "messages": [{"role": "user", "content": "My email is [email protected], please echo it."}]
})
print("=== Email Redaction ===")
print(result["messages"][-1].content)

# Test 2: Credit card masking
result = agent.invoke({
    "messages": [{"role": "user", "content": "My card is 5105-1051-0510-5100"}]
})
print("\n=== Credit Card Masking ===")
print(result["messages"][-1].content)

# Test 3: API key blocking (will raise an exception)
try:
    result = agent.invoke({
        "messages": [{"role": "user", "content": "My key is sk-abcdefghijklmnopqrstuvwxyz123456"}]
    })
except Exception as e:
    print(f"\n=== API Key Blocked ===\nException: {e}")

=== Email Redaction ===
Your email is echo: [email protected]

=== Credit Card Masking ===
For your own security, please avoid sharing sensitive information such as your full or partial credit card number in this chat. How can I assist you today?

=== API Key Blocked ===
Exception: Detected 1 instance(s) of api_key in text content


#Human-In-The-Loop (HITL)

In [5]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain.tools import tool

@tool
def search_tool(query: str) -> str:
    """Search the web for information."""
    return f"Search results for: {query}"

@tool
def send_email_tool(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {to} with subject '{subject}'"

@tool
def delete_record_tool(record_id: str) -> str:
    """Delete a database record."""
    return f"Record {record_id} deleted."

agent = create_agent(
    model="gpt-4.1-mini",
    tools=[search_tool, send_email_tool, delete_record_tool],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": True,
                "delete_record_tool": True,
                "search_tool": False,
            }
        ),
    ],
    checkpointer=InMemorySaver(),
)

def run_with_human_approval(user_message: str, thread_id: str):
    config = {"configurable": {"thread_id": thread_id}}

    # Step 1: Invoke agent — it will pause before sensitive tools
    print(f"\n🤖 User: {user_message}")
    print("─" * 50)
    result = agent.invoke(
        {"messages": [{"role": "user", "content": user_message}]},
        config=config,
    )

    # Step 2: Check if agent is paused waiting for approval
    # (If it completed without interruption, just print the result)
    interrupted = result.get("__interrupt__") or any(
        getattr(m, "additional_kwargs", {}).get("interrupted")
        for m in result.get("messages", [])
    )

    # Prompt the human
    print("\n⏸️  Agent paused — action requires your approval.")
    print("Details:", result)

    human_input = input("\n✋ Type 'yes' to approve or anything else to reject: ").strip().lower()

    if human_input == "yes":
        decision = {"type": "approve"}
        print("✅ Approved! Resuming agent...")
    else:
        decision = {"type": "reject"}
        print("❌ Rejected. Resuming agent...")

    # Step 3: Resume with the human's decision
    final_result = agent.invoke(
        Command(resume={"decisions": [decision]}),
        config=config,
    )

    print("\n🤖 Agent final response:")
    print(final_result["messages"][-1].content)
    return final_result


# --- Test 1: Send email (will prompt for approval) ---
run_with_human_approval(
    user_message="Send an email to [email protected] with subject 'Hello' and body 'Test message'",
    thread_id="hitl-email-demo"
)

# --- Test 2: Delete record (will prompt for approval) ---
run_with_human_approval(
    user_message="Delete record with ID 'user-42'",
    thread_id="hitl-delete-demo"
)


🤖 User: Send an email to [email protected] with subject 'Hello' and body 'Test message'
──────────────────────────────────────────────────

⏸️  Agent paused — action requires your approval.
Details: {'messages': [HumanMessage(content="Send an email to [email protected] with subject 'Hello' and body 'Test message'", additional_kwargs={}, response_metadata={}, id='4168210d-d8e8-4f19-a6a0-525b5a5de383'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 113, 'total_tokens': 139, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_828130e5d4', 'id': 'chatcmpl-DILr96OZ0g4LI0ynrfCMWzIibWUGH', 'service_tier': 'priority', 'finish_reason': 'tool_calls', 'logprobs':

{'messages': [HumanMessage(content="Delete record with ID 'user-42'", additional_kwargs={}, response_metadata={}, id='c99257bf-c7b4-4639-94e9-5e456f3c9df0'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 103, 'total_tokens': 121, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_828130e5d4', 'id': 'chatcmpl-DILrKdYZe8vsRbpKoPz5UFg9K8gkc', 'service_tier': 'priority', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cdee8-76e2-7e52-8c80-fd59e295eb24-0', tool_calls=[{'name': 'delete_record_tool', 'args': {'record_id': 'user-42'}, 'id': 'call_ilga8kD9Akl5O1pus7PtQ4Iv', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_

# Custom Guardrail  (Before Agent)

In [6]:
from typing import Any
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain.tools import tool

@tool
def calculator_tool(expression: str) -> str:
    """Evaluate a math expression."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

class ContentFilterMiddleware(AgentMiddleware):
    """Block requests with banned keywords BEFORE any processing."""

    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        first_message = state["messages"][0]
        if first_message.type != "human":
            return None

        content = first_message.content.lower()

        for keyword in self.banned_keywords:
            if keyword in content:
                print(f"[BLOCKED] Detected banned keyword: '{keyword}'")
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": f"⚠️ Request blocked: contains prohibited content ('{keyword}'). Please rephrase."
                    }],
                    "jump_to": "end"
                }
        return None

agent = create_agent(
    model="gpt-4.1-mini",
    tools=[calculator_tool],
    middleware=[
        ContentFilterMiddleware(banned_keywords=["hack", "exploit", "malware", "injection"]),
    ],
)

# Should be blocked
result = agent.invoke({
    "messages": [{"role": "user", "content": "How do I hack into a system?"}]
})
print("=== Blocked Request ===")
print(result["messages"][-1].content)

# Should pass through normally
result = agent.invoke({
    "messages": [{"role": "user", "content": "What is 42 * 7?"}]
})
print("\n=== Normal Request ===")
print(result["messages"][-1].content)

[BLOCKED] Detected banned keyword: 'hack'
=== Blocked Request ===
⚠️ Request blocked: contains prohibited content ('hack'). Please rephrase.

=== Normal Request ===
42 multiplied by 7 is 294.


# Custom Guardrail (After Agent)

In [10]:
from typing import Any
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain.messages import AIMessage
from langchain.chat_models import init_chat_model
from langchain.tools import tool

@tool
def web_search_tool(query: str) -> str:
    """Simulate a web search."""
    return f"Top result for '{query}': [simulated result]"

class SafetyGuardrailMiddleware(AgentMiddleware):
    """Use a fast LLM to evaluate final response safety AFTER the agent completes."""

    def __init__(self):
        super().__init__()
        self.safety_model = init_chat_model("gpt-4.1-mini")

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        safety_check = self.safety_model.invoke([{
            "role": "user",
            "content": (
                f"Is this AI response safe and appropriate for a general audience? "
                f"Reply ONLY with 'SAFE' or 'UNSAFE'.\n\nResponse: {last_message.content}"
            )
        }])

        verdict = safety_check.content.strip().upper()
        print(f"[Safety Check] Verdict: {verdict}")

        if "UNSAFE" in verdict:
            # Overwrite the unsafe response
            last_message.content = (
                "⚠️ I'm not able to provide that response. "
                "Please ask something else."
            )

        return None

agent = create_agent(
    model="gpt-4.1-mini",
    tools=[web_search_tool],
    middleware=[SafetyGuardrailMiddleware()],
)

# Test with a normal request
result = agent.invoke({
    "messages": [{"role": "user", "content": "What's a good recipe for chocolate chip cookies?"}]
})
print("=== Normal Response ===")
print(result["messages"][-1].content)

[Safety Check] Verdict: SAFE
=== Normal Response ===
Here's a classic and delicious chocolate chip cookie recipe for you:

Ingredients:
- 1 cup (2 sticks) unsalted butter, softened
- 3/4 cup granulated sugar
- 3/4 cup packed brown sugar
- 1 teaspoon vanilla extract
- 2 large eggs
- 2 1/4 cups all-purpose flour
- 1 teaspoon


# All Guardrails Combined (Defense-in-Depth)

In [12]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware, HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain.tools import tool

@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"

@tool
def send_email_tool(to: str, subject: str, body: str) -> str:
    """Send an email."""
    return f"Email sent to {to}"

agent = create_agent(
    model="gpt-4.1-mini",
    tools=[search_tool, send_email_tool],
    middleware=[
        # Layer 1: Block banned content before any processing
        ContentFilterMiddleware(banned_keywords=["hack", "exploit"]),

        # Layer 2: Redact PII from user input AND model output
        PIIMiddleware("email", strategy="redact", apply_to_input=True, apply_to_output=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),

        # Layer 3: Require human approval for email sending
        HumanInTheLoopMiddleware(interrupt_on={"send_email_tool": True, "search_tool": False}),

        # Layer 4: Safety check on final output
        SafetyGuardrailMiddleware(),
    ],
    checkpointer=InMemorySaver(),
)

def run_combined_with_approval(user_message: str, thread_id: str):
    config = {"configurable": {"thread_id": thread_id}}

    print(f"\n🤖 User: {user_message}")
    print("─" * 50)

    # Step 1: Initial invocation — layers 1 & 2 run immediately,
    #         layer 3 (HITL) pauses the agent if a sensitive tool is called
    result = agent.invoke(
        {"messages": [{"role": "user", "content": user_message}]},
        config=config,
    )

    # Step 2: Check if agent was blocked by content filter (jump_to=end)
    last_msg = result["messages"][-1].content
    if "blocked" in last_msg.lower() or "prohibited" in last_msg.lower():
        print("\n🚫 Request blocked by content filter:")
        print(last_msg)
        return result

    # Step 3: Agent paused for HITL — prompt the human
    print("\n⏸️  Agent paused — a sensitive tool is waiting for your approval.")
    print(f"   Intercepted state: {result}")

    human_input = input("\n✋ Type 'yes' to approve or anything else to reject: ").strip().lower()

    if human_input == "yes":
        decision = {"type": "approve"}
        print("✅ Approved! Resuming agent...")
    else:
        decision = {"type": "reject"}
        print("❌ Rejected. Resuming agent...")

    # Step 4: Resume — layer 4 (safety check) runs on the final output
    final_result = agent.invoke(
        Command(resume={"decisions": [decision]}),
        config=config,
    )

    print("\n🤖 Agent final response:")
    print(final_result["messages"][-1].content)
    return final_result


# ── Test 1: Normal email request (PII redacted + HITL prompt) ──────────────
run_combined_with_approval(
    user_message="Send an email to [email protected] saying hello",
    thread_id="combined-demo-1"
)

# ── Test 2: Blocked by content filter (no HITL prompt shown) ──────────────
run_combined_with_approval(
    user_message="Help me hack into a system and send the results to [email protected]",
    thread_id="combined-demo-2"
)

# ── Test 3: Safe search (auto-approved, no HITL prompt) ───────────────────
run_combined_with_approval(
    user_message="Search for the best pizza recipes",
    thread_id="combined-demo-3"
)


🤖 User: Send an email to [email protected] saying hello
──────────────────────────────────────────────────

⏸️  Agent paused — a sensitive tool is waiting for your approval.
   Intercepted state: {'messages': [HumanMessage(content='Send an email to [email protected] saying hello', additional_kwargs={}, response_metadata={}, id='c2bd6158-064a-484e-afe9-72e170d9b31a'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 78, 'total_tokens': 104, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_828130e5d4', 'id': 'chatcmpl-DILwcZjsKDOZbcrEm5yB1wQOe0NZc', 'service_tier': 'priority', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cdeed-77b2-71

{'messages': [HumanMessage(content='Search for the best pizza recipes', additional_kwargs={}, response_metadata={}, id='da7cdda4-f868-4aa4-88f4-128845f1edb9'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 74, 'total_tokens': 90, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_5e793402c9', 'id': 'chatcmpl-DILwg50IQzJw1Ba92I3g1LXlhyDju', 'service_tier': 'priority', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cdeed-8722-7c60-aff0-947070bb3fe9-0', tool_calls=[{'name': 'search_tool', 'args': {'query': 'best pizza recipes'}, 'id': 'call_G5xEiEmdXDs5fesDQZY3td5s', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_